# M5: Runtime Layer Separation

M4's `Engine._run` did everything in one call: build the tensor, run `model.generate()`, pick tokens, and decide when to stop. M5 splits that into three parts with one job each.

```text
Engine                 tokenizer + wiring, no model or queue logic
  ├─ Scheduler         who runs, when a request finishes      (scheduler.py)
  └─ ModelRunner       device + one forward step: tokens -> logits   (model_runner.py)
       └─ Sampler      logits -> next token id (greedy)      (sampler.py)
```

![M5 architecture](m5_arch.jpeg)

The biggest change: `model.generate()` is gone. The scheduler now owns the decode loop, and one loop iteration makes **one token**, not one whole request.

```text
M4: one iteration = run_batch -> engine._run(req) -> model.generate()    (whole request)
M5: one iteration = run_batch -> forward(req) -> sample(logits)          (one token)
                    process_batch_result: append token, check EOS / max_new_tokens
```

This notebook checks:

- One forward step by hand: what logits look like, and how greedy picks a token
- The hand-made loop gives exactly the same tokens as HF greedy `generate()`
- Without a KV cache, each step recomputes the whole sequence and gets slower
- The scheduler runs one token per iteration, requests still FIFO, and new arrivals are picked up between tokens
- EOS stops a request early with `finish_reason="stop"`
- A failing request gets `500`, and the server keeps serving

## Setup

Same as M4: start uvicorn in a background thread so the notebook can act as the client. `Engine()` builds the `ModelRunner` and `Scheduler`, and starts the scheduler thread.

Port 30000 is SGLang's default. Stop any `python server.py` you started in a terminal first, or the port is taken.

In [1]:
import json
import threading
import time

import httpx
import torch
import uvicorn

from engine import Engine
from req import Req
from server import create_app

engine = Engine()
runner = engine.model_runner
tok = engine.tokenizer
print(runner.device, "| eos_token_ids:", {i: tok.decode([i]) for i in runner.eos_token_ids})

server = uvicorn.Server(uvicorn.Config(create_app(engine), host="127.0.0.1", port=30000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

client = httpx.Client(base_url="http://127.0.0.1:30000", timeout=300)


def show(resp):
    print(resp.status_code, resp.reason_phrase)
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print(repr(resp.text))  # not every error body is JSON

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps | eos_token_ids: {151643: '<|endoftext|>', 151645: '<|im_end|>'}


## One Forward Step by Hand

Use the `ModelRunner` directly, with no scheduler. This `Req` is never submitted, so nothing else touches it.

`forward(req)` feeds `input_ids + output_ids` as a `[1, seq_len]` tensor to `self.model(...)`. The model returns logits of shape `[1, seq_len, vocab_size]`: position `i` scores the token at position `i + 1`. Only the last position predicts a new token, so `forward` returns `logits[0, -1]`, shape `[vocab_size]`.

Logits are raw scores, not probabilities. `softmax` turns them into probabilities that sum to 1. Greedy just takes the `argmax`.

In [2]:
req = Req(rid="manual", prompt="The capital of France is", input_ids=tok.encode("The capital of France is"), max_new_tokens=8)

logits = runner.forward(req)
print("logits:", tuple(logits.shape), logits.dtype, "| vocab_size:", runner.model.config.vocab_size)

probs = torch.softmax(logits.float(), dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"{i.item():>7}  {tok.decode([i.item()])!r:<12} logit={logits[i].item():6.2f}  prob={p.item():.3f}")

print("sample() picks:", runner.sample(logits, req), "(the argmax)")

logits: (151936,) torch.bfloat16 | vocab_size: 151936
  12095  ' Paris'     logit= 17.50  prob=0.657
   7407  ' located'   logit= 14.38  prob=0.029
    279  ' the'       logit= 14.06  prob=0.021
   1112  '...'        logit= 13.75  prob=0.015
  30743  ' ____'      logit= 13.69  prob=0.015
sample() picks: 12095 (the argmax)


## The Decode Loop, Written Out

This is what the scheduler does, minus the queues: `forward`, `sample`, append, check for a stop. Each step's input is the prompt plus every token generated so far, so the sequence grows by one per step.

In [3]:
req.output_ids.clear()
for step in range(req.max_new_tokens):
    seq_len = len(req.input_ids) + len(req.output_ids)
    token = runner.sample(runner.forward(req), req)
    req.output_ids.append(token)
    print(f"step {step}: seq_len={seq_len:>2} -> {token:>6} {tok.decode([token])!r}")
    if token in runner.eos_token_ids:
        break

print(repr(req.prompt + tok.decode(req.output_ids)))

step 0: seq_len= 5 ->  12095 ' Paris'
step 1: seq_len= 6 ->     13 '.'
step 2: seq_len= 7 ->    576 ' The'
step 3: seq_len= 8 ->   6722 ' capital'
step 4: seq_len= 9 ->    315 ' of'
step 5: seq_len=10 ->   9625 ' France'
step 6: seq_len=11 ->    374 ' is'
step 7: seq_len=12 ->   1083 ' also'
'The capital of France is Paris. The capital of France is also'


## Same Tokens as HF Greedy `generate()`

`generate()` runs the same loop internally. With `do_sample=False` (greedy), `use_cache=False` (no KV cache, like us), and `eos_token_id=None` (never stop early), it must give exactly our tokens. `test_step_loop_matches_hf_greedy_generate` checks the same thing on a tiny random model.

M4 used `do_sample=True`, so the same prompt gave different text each run. M5's greedy sampler is deterministic.

In [4]:
with torch.inference_mode():
    hf = runner.model.generate(
        torch.tensor([req.input_ids], device=runner.device),
        max_new_tokens=len(req.output_ids),
        do_sample=False,
        use_cache=False,
        eos_token_id=None,
    )[0, len(req.input_ids) :].tolist()

print("ours:", req.output_ids)
print("hf  :", hf)
print("match:", req.output_ids == hf)

ours: [12095, 13, 576, 6722, 315, 9625, 374, 1083]
hf  : [12095, 13, 576, 6722, 315, 9625, 374, 1083]
match: True


## No KV Cache: Every Step Recomputes Everything

`forward` runs the model over the whole sequence each step, even though only the last position is new. So a step gets slower as the sequence grows. Time one `forward` call at a few lengths.

`torch.mps.synchronize()` / `torch.cuda.synchronize()` waits for the GPU to finish, so the timer measures the real compute, not just the kernel launch.

In [5]:
def sync():
    if runner.device == "cuda":
        torch.cuda.synchronize()
    elif runner.device == "mps":
        torch.mps.synchronize()


def time_forward(seq_len, repeat=3):
    r = Req(rid="t", prompt="", input_ids=[tok.encode(" hello")[0]] * seq_len, max_new_tokens=1)
    runner.forward(r)  # warm up
    sync()
    start = time.perf_counter()
    for _ in range(repeat):
        runner.forward(r)
    sync()
    return (time.perf_counter() - start) / repeat * 1000


for n in [16, 128, 512, 1024, 2048]:
    print(f"seq_len={n:>5}: {time_forward(n):7.1f} ms per step")

seq_len=   16:    44.3 ms per step
seq_len=  128:   145.1 ms per step
seq_len=  512:   517.5 ms per step
seq_len= 1024:  1043.7 ms per step
seq_len= 2048:  2213.0 ms per step


Step time grows about linearly with sequence length. Generating N tokens pays this at every step, so the total cost grows roughly with N². The KV cache milestone removes the recompute: each step then feeds only the newest token.

## Record Each Scheduler Step

Wrap `runner.forward` to log which request each step serves. Setting `forward` on the instance shadows the class method. `del runner.forward` restores it.

The scheduler holds the same `runner` object and looks up `forward` at every call, so it sees the wrapper.

In [6]:
steps = []  # prompt letter of the request each forward call served
original_forward = type(runner).forward


def logged_forward(req):
    steps.append(req.prompt[:1])
    return original_forward(runner, req)


runner.forward = logged_forward

## One Token per Iteration, Still FIFO

Send A, B, and C from three threads, 0.2 s apart. Each loop iteration now calls `forward` once and adds one token. Capacity is still one, so A runs all its steps, then B, then C. The output is the same as M4; only the step size changed.

In [7]:
prompts = ["A: Write a story about a dragon.", "B: Write a poem about the sea.", "C: Explain what a GPU does."]
results = {}


def post(prompt, max_new_tokens=16):
    results[prompt] = client.post("/generate", json={"text": prompt, "sampling_params": {"max_new_tokens": max_new_tokens}})


steps.clear()
threads = [threading.Thread(target=post, args=(p,)) for p in prompts]
for t in threads:
    t.start()
    time.sleep(0.2)
for t in threads:
    t.join()

for p in prompts:
    r = results[p].json()
    print(p[:1], results[p].status_code, r["meta_info"]["finish_reason"], r["meta_info"]["completion_tokens"], "tokens")

print("forward calls:", len(steps))
print("".join(steps))

A 200 length 16 tokens
B 200 length 16 tokens
C 200 length 16 tokens
forward calls: 48
AAAAAAAAAAAAAAAABBBBBBBBBBBBBBBBCCCCCCCCCCCCCCCC


## Look Inside the Scheduler While A Runs

M4 ran A in one long step, so B and C always sat in `recv_queue` until A finished.

M5's steps are one token long. `recv_requests()` runs before every step, so B and C move to `waiting_queue` while A is still generating. They still wait for A, but the scheduler already sees them. This is what lets later milestones add them to a running batch.

In [8]:
steps.clear()
threads = [threading.Thread(target=post, args=(p, 64)) for p in prompts]
for t in threads:
    t.start()
    time.sleep(0.2)

s = engine.scheduler
running = s.running_req
print("running_req  :", running.prompt[:1] if running else None, f"({len(running.output_ids)} tokens so far)" if running else "")
print("waiting_queue:", [r.prompt[:1] for r in s.waiting_queue])
print("recv_queue   :", s.recv_queue.qsize(), "request(s)")

for t in threads:
    t.join()
print("after        :", s.running_req, list(s.waiting_queue), s.recv_queue.qsize())

running_req  : A (13 tokens so far)
waiting_queue: ['B', 'C']
recv_queue   : 0 request(s)
after        : None [] 0


## EOS Stops a Request Early

`process_batch_result` checks every new token against `eos_token_ids` before checking `max_new_tokens`. Qwen3 has two EOS tokens, `<|im_end|>` (end of a chat turn) and `<|endoftext|>`. A raw prompt rarely ends a turn, so wrap the question in the chat template. `enable_thinking=False` skips Qwen3's `<think>` block.

The output ends with the EOS token id. `text` hides it, because `decode` skips special tokens.

In [9]:
chat = tok.apply_chat_template(
    [{"role": "user", "content": "Reply with one word: what color is the sky?"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
resp = client.post("/generate", json={"text": chat, "sampling_params": {"max_new_tokens": 64}})
show(resp)
last = resp.json()["output_ids"][-1]
print("last token:", last, repr(tok.decode([last])), "| is EOS:", last in runner.eos_token_ids)

200 OK
{
  "text": "sky",
  "output_ids": [
    26684,
    151645
  ],
  "meta_info": {
    "id": "5630bc71bd724d83958a9834efa95e80",
    "finish_reason": "stop",
    "prompt_tokens": 23,
    "completion_tokens": 2
  }
}
last token: 151645 '<|im_end|>' | is EOS: True


## An Idle Scheduler Uses No CPU

Still true in M5: with nothing running and nothing waiting, `recv_requests()` blocks on `recv_queue.get()`. A busy loop would show about 1 s here.

In [10]:
start = time.process_time()
time.sleep(1)
print(f"CPU used while idle for 1 s: {time.process_time() - start:.3f}s")

CPU used while idle for 1 s: 0.009s


## A Failing Request Gets 500, and the Server Keeps Serving

Make `forward` raise. The error now happens in the middle of a request, not at the start of one big call. `event_loop` catches it and stores it on `req.error`. `process_batch_result` gets `result=None`, clears `running_req`, and sets `req.done`. Without clearing `running_req`, the scheduler would keep picking the broken request forever.

In [11]:
def failing_forward(req):
    raise RuntimeError("model crashed")


runner.forward = failing_forward
# uvicorn drops the connection after an unhandled error; close it so the next request opens a fresh one
show(client.post("/generate", json={"text": "Hello"}, headers={"Connection": "close"}))

del runner.forward  # back to the real model
show(client.post("/generate", json={"text": "Hello", "sampling_params": {"max_new_tokens": 8}}))

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/protocols/http/h11_impl.py", line 411, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/middl

500 Internal Server Error
'Internal Server Error'
200 OK
{
  "text": " Answer! I'm a bit confused about",
  "output_ids": [
    21806,
    0,
    358,
    2776,
    264,
    2699,
    21815,
    911
  ],
  "meta_info": {
    "id": "f8d04bebf4cf484e89c8dd5eb0e5ceb2",
    "finish_reason": "length",
    "prompt_tokens": 1,
    "completion_tokens": 8
  }
}


## Summary

| | M4 | M5 |
|---|---|---|
| Who owns the decode loop | HF `model.generate()` | our `Scheduler` |
| One loop iteration | a whole request | one token |
| Model call | `model.generate(...)` | `model(input_ids).logits[0, -1]` in `ModelRunner.forward` |
| Token choice | HF default sampling (`do_sample=True`) | `Sampler`: greedy `argmax`, deterministic |
| Stop check | read the last token after `generate()` | `process_batch_result`, every step |
| New arrivals while running | stay in `recv_queue` | moved to `waiting_queue` between tokens |
| KV cache | none | none, every step recomputes the whole sequence |

Owning the loop is the point. Streaming, abort, batching, and per-request sampling all need to act between tokens, and `generate()` gives no place to do that.

**Compare with SGLang**: `ModelRunner.forward()` and `ModelRunner.sample()` in `model_executor/model_runner.py`, and `Sampler` in `layers/sampler.py`. SGLang's `run_batch()` and `process_batch_result()` in `managers/scheduler.py` have the same split as ours, but work on a whole batch per step.

## Shutdown

In [12]:
client.close()
server.should_exit = True